# Lab 03: Memory & RAG — Give Your Agent Knowledge and Memory

## 🎯 What We're Building

| Stage | What | What You Learn |
|-------|------|----------------|
| **Stage 1** | Agent without RAG — see it fail | Why LLMs hallucinate on YOUR data |
| **Stage 2** | Build a RAG pipeline | Embed docs → index in Azure AI Search → retrieve |
| **Stage 3** | RAG agent with LangGraph | Agent that searches docs before answering |
| **Stage 4** | Persistent memory with Cosmos DB | Conversations that survive restarts |
| **Stage 5** | The complete agent | RAG + Memory combined |

> **Prerequisites:** Complete [Lab 01](../lab-01-react-agent/README.md) and [Lab 02](../lab-02-model-routing/README.md). Read the [README.md](README.md).

---

## 🔧 Setup

This lab uses several Azure services:
- **Azure OpenAI** — GPT-4.1 (reasoning) + text-embedding-3-large (embeddings)
- **Azure AI Search** — Vector index for document search
- **Azure Cosmos DB** — Persistent memory for conversations

All were deployed in Lab 00.

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI

# ──────────────────────────────────────────────────────
# Load all Azure connection strings
# ──────────────────────────────────────────────────────
load_dotenv("../.env")

# Azure OpenAI client (for both chat and embeddings)
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
)

MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-41")
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_EMBEDDING", "text-embedding-3-large")

print(f"✅ Azure OpenAI connected")
print(f"   Chat model: {MODEL}")
print(f"   Embedding model: {EMBEDDING_MODEL}")
print(f"   Search endpoint: {os.getenv('AZURE_SEARCH_ENDPOINT')}")
print(f"   Cosmos DB: {os.getenv('AZURE_COSMOS_ENDPOINT', 'not configured')}")

✅ Azure OpenAI connected
   Chat model: gpt-41
   Embedding model: text-embedding-3-large
   Search endpoint: https://search-agentlabs-fk25xdiuwudo4.search.windows.net
   Cosmos DB: https://cosmos-agentlabs-fk25xdiuwudo4.documents.azure.com:443/


---

# 🏗️ STAGE 1: See the Agent Fail (No RAG)

Let's ask our agent questions about **Acme Corp** — a fictional company with
specific policies, pricing, and SLAs.

The LLM has NEVER seen these documents. Watch what happens when it tries to
answer anyway — it will **hallucinate** (make up plausible-sounding but wrong answers).

### Why does hallucination happen?

The LLM was trained on internet data. When asked about specific company details,
it doesn't say "I don't know." Instead, it generates a response based on
**general patterns** it learned — which is often wrong for YOUR specific data.

In [2]:
# ──────────────────────────────────────────────────────
# Ask about Acme Corp WITHOUT giving the agent any docs
#
# Watch carefully: the LLM will make up answers that
# SOUND plausible but are WRONG.
#
# This is exactly why RAG exists.
# ──────────────────────────────────────────────────────

questions = [
    "What is Acme Corp's refund policy for enterprise customers?",
    "How many vacation days do Acme Corp employees get?",
    "What is the price of Acme Corp's Professional plan?",
    "What is the uptime SLA for Acme Corp Enterprise customers?",
]

print("📊 STAGE 1: Agent WITHOUT RAG (no access to company docs)")
print("=" * 60)

for q in questions:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": q}],
        max_tokens=150
    )
    answer = response.choices[0].message.content
    print(f"\n❓ {q}")
    print(f"🤖 {answer[:200]}...")
    print(f"⚠️  Is this correct? We don't know — the LLM is guessing!")

📊 STAGE 1: Agent WITHOUT RAG (no access to company docs)

❓ What is Acme Corp's refund policy for enterprise customers?
🤖 I’m sorry, but I don’t have access to Acme Corp’s specific refund policy for enterprise customers. Refund policies vary between companies and may depend on your contract or agreement with Acme Corp.

...
⚠️  Is this correct? We don't know — the LLM is guessing!

❓ How many vacation days do Acme Corp employees get?
🤖 I’m sorry, but I don’t have specific information about Acme Corp’s vacation policy. Vacation days can vary widely across companies and locations. If you’re an employee or job applicant, I recommend ch...
⚠️  Is this correct? We don't know — the LLM is guessing!

❓ What is the price of Acme Corp's Professional plan?
🤖 I'm sorry, but I don't have real-time access to Acme Corp's pricing. To find the current price of Acme Corp's Professional plan, please visit their [official website](#), or contact their sales team d...
⚠️  Is this correct? We don't know — 

### 🤔 What did you notice?

The LLM gave answers that **sound reasonable** but are likely **completely wrong**:
- It doesn't know Acme Corp's actual refund timeline (14 days standard, 30 days enterprise)
- It doesn't know the actual vacation days (22 days)
- It doesn't know the actual pricing ($79/month for Professional)
- It doesn't know the actual SLA (99.9% for Enterprise)

**This is hallucination** — and it's exactly what RAG fixes.

---

# 🏗️ STAGE 2: Build a RAG Pipeline

Now let's build the RAG pipeline that will give our agent access to Acme Corp's documents.

### The RAG Pipeline in 4 Steps

```
1. LOAD      → Read the markdown documents from disk
2. CHUNK     → Split each document into smaller pieces (~500 tokens each)
3. EMBED     → Convert each chunk to a vector (3072 numbers) using Azure OpenAI
4. INDEX     → Store vectors + text in Azure AI Search
```

After indexing, we can **search** by meaning — not just keywords.

### Why chunk?

LLMs have limited context windows. If your document is 50 pages, you can't send
the whole thing. Instead, you:
1. Split into chunks (~500 tokens each)
2. Find the 3-5 most relevant chunks for the question
3. Send only those chunks as context

> 📚 For deep chunking strategies, see [RAG Workshop Module 4](https://github.com/roie9876/RAG-WorkShop/tree/main/modules/module-4-chunking)

In [3]:
# ──────────────────────────────────────────────────────
# STEP 1: Load documents
#
# We have 4 sample documents about Acme Corp:
# - Refund & Returns Policy
# - Employee Benefits & Leave Policy
# - Product Pricing & Plans
# - Service Level Agreement (SLA)
# ──────────────────────────────────────────────────────

data_dir = Path("data")
documents = []

for file_path in sorted(data_dir.glob("*.md")):
    content = file_path.read_text()
    documents.append({
        "filename": file_path.name,
        "content": content
    })
    print(f"📄 Loaded: {file_path.name} ({len(content)} chars)")

print(f"\n✅ Loaded {len(documents)} documents")

📄 Loaded: employee-benefits.md (1618 chars)
📄 Loaded: pricing-plans.md (1809 chars)
📄 Loaded: refund-policy.md (1373 chars)
📄 Loaded: sla-agreement.md (1713 chars)

✅ Loaded 4 documents


In [4]:
# ──────────────────────────────────────────────────────
# STEP 2: Chunk the documents
#
# We split each document into chunks of ~500 characters
# with 100 characters overlap.
#
# Why overlap? So that sentences at chunk boundaries
# aren't cut in half. The overlap means each chunk
# shares some text with its neighbors.
#
# In production, you'd use smarter chunking (by section,
# by paragraph, or semantic chunking). For this lab,
# simple character splitting works fine.
# ──────────────────────────────────────────────────────

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

chunks = []

for doc in documents:
    text = doc["content"]
    # Split into chunks with overlap
    start = 0
    chunk_idx = 0
    while start < len(text):
        end = start + CHUNK_SIZE
        chunk_text = text[start:end]
        
        chunks.append({
            "id": f"{doc['filename']}-chunk-{chunk_idx}",
            "content": chunk_text,
            "source": doc["filename"],
            "chunk_index": chunk_idx,
        })
        
        start += CHUNK_SIZE - CHUNK_OVERLAP
        chunk_idx += 1

print(f"✅ Created {len(chunks)} chunks from {len(documents)} documents")
print(f"   Chunk size: {CHUNK_SIZE} chars, overlap: {CHUNK_OVERLAP} chars")
print(f"\n📄 Chunks per document:")
for doc in documents:
    doc_chunks = [c for c in chunks if c["source"] == doc["filename"]]
    print(f"   {doc['filename']}: {len(doc_chunks)} chunks")

✅ Created 19 chunks from 4 documents
   Chunk size: 500 chars, overlap: 100 chars

📄 Chunks per document:
   employee-benefits.md: 5 chunks
   pricing-plans.md: 5 chunks
   refund-policy.md: 4 chunks
   sla-agreement.md: 5 chunks


In [5]:
# ──────────────────────────────────────────────────────
# STEP 3: Generate embeddings
#
# An "embedding" converts text into a vector of numbers
# (3072 numbers for text-embedding-3-large).
#
# Texts with similar MEANING produce similar vectors.
# This is how we search by meaning, not just keywords.
#
# Example:
#   "refund policy" ≈ "how to get money back" (similar vectors)
#   "refund policy" ≠ "pizza recipe" (different vectors)
# ──────────────────────────────────────────────────────

print("🔢 Generating embeddings for all chunks...")

# Embed all chunks in one batch call (faster than one-by-one)
batch_texts = [chunk["content"] for chunk in chunks]

embedding_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=batch_texts
)

# Attach embeddings to chunks
for i, chunk in enumerate(chunks):
    chunk["embedding"] = embedding_response.data[i].embedding

dim = len(chunks[0]["embedding"])
print(f"✅ Generated embeddings for {len(chunks)} chunks")
print(f"   Vector dimensions: {dim}")
print(f"   Each chunk is now a point in {dim}-dimensional space!")

🔢 Generating embeddings for all chunks...
✅ Generated embeddings for 19 chunks
   Vector dimensions: 3072
   Each chunk is now a point in 3072-dimensional space!


In [6]:
# ──────────────────────────────────────────────────────
# STEP 4: Index in Azure AI Search
#
# Azure AI Search stores our chunks + vectors and lets
# us search by meaning (vector search), keywords
# (full-text search), or both (hybrid search).
#
# We'll create:
# 1. An index schema (defines fields and search modes)
# 2. Upload all chunks with their embeddings
# ──────────────────────────────────────────────────────

from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType,
    SimpleField, SearchableField,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    SearchIndex, SemanticConfiguration, SemanticSearch, SemanticPrioritizedFields, SemanticField
)
from azure.core.credentials import AzureKeyCredential

SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
SEARCH_KEY = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME = "agent-labs-rag"

credential = AzureKeyCredential(SEARCH_KEY)

# Create the index schema
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),  # Full-text searchable
    SimpleField(name="source", type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="chunk_index", type=SearchFieldDataType.Int32),
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=3072,
        vector_search_profile_name="default-profile"
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="default-algorithm")],
    profiles=[VectorSearchProfile(name="default-profile", algorithm_configuration_name="default-algorithm")],
)

semantic_config = SemanticConfiguration(
    name="default-semantic",
    prioritized_fields=SemanticPrioritizedFields(content_fields=[SemanticField(field_name="content")])
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=SemanticSearch(configurations=[semantic_config])
)

# Create or update the index
index_client.create_or_update_index(index)
print(f"✅ Index '{INDEX_NAME}' created/updated")

# Upload chunks
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=credential)

# Prepare documents for upload (Azure Search format)
search_docs = []
for chunk in chunks:
    search_docs.append({
        "id": chunk["id"].replace(".", "-"),  # IDs can't have dots
        "content": chunk["content"],
        "source": chunk["source"],
        "chunk_index": chunk["chunk_index"],
        "embedding": chunk["embedding"],
    })

result = search_client.upload_documents(documents=search_docs)
succeeded = sum(1 for r in result if r.succeeded)
print(f"✅ Uploaded {succeeded}/{len(search_docs)} chunks to Azure AI Search")
print(f"\n🎉 RAG pipeline complete! Documents are now searchable.")

✅ Index 'agent-labs-rag' created/updated
✅ Uploaded 19/19 chunks to Azure AI Search

🎉 RAG pipeline complete! Documents are now searchable.


### Let's Test the Search

Before building the agent, let's verify that search works.
We'll try three search modes:
- **Vector search** — search by meaning (embedding similarity)
- **Keyword search** — traditional full-text search
- **Hybrid search** — both combined (usually best)

In [7]:
# ──────────────────────────────────────────────────────
# Test search: find relevant chunks for a question
#
# We embed the question and search for similar chunks.
# This is how the agent will find relevant documents.
# ──────────────────────────────────────────────────────

from azure.search.documents.models import VectorizedQuery

def search_documents(query: str, top_k: int = 3) -> list[dict]:
    """Search the RAG index using hybrid search (vector + keyword)."""
    # Embed the query
    query_embedding = client.embeddings.create(
        model=EMBEDDING_MODEL, input=query
    ).data[0].embedding
    
    # Hybrid search: vector + keyword
    results = search_client.search(
        search_text=query,  # Keyword search
        vector_queries=[VectorizedQuery(
            vector=query_embedding,
            k_nearest_neighbors=top_k,
            fields="embedding"
        )],
        top=top_k,
    )
    
    return [{"content": r["content"], "source": r["source"], "score": r["@search.score"]} for r in results]

# Test it!
test_query = "What is the refund policy for enterprise customers?"
results = search_documents(test_query)

print(f"🔍 Query: {test_query}")
print(f"\n📋 Top {len(results)} results:")
for i, r in enumerate(results):
    print(f"\n  [{i+1}] Source: {r['source']} (score: {r['score']:.4f})")
    print(f"      {r['content'][:150]}...")

k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizedQuery'> and will be ignored


🔍 Query: What is the refund policy for enterprise customers?

📋 Top 3 results:

  [1] Source: refund-policy.md (score: 0.0333)
      # Acme Corp — Refund & Returns Policy
**Document Version:** 3.2 | **Last Updated:** January 2026 | **Owner:** Legal Department

## 1. Standard Refund ...

  [2] Source: refund-policy.md (score: 0.0323)
      nd terms:
- Full refund within **30 calendar days** of contract signing
- After 30 days: prorated refund minus a 10% early termination fee
- Annual co...

  [3] Source: sla-agreement.md (score: 0.0313)
      ble for Enterprise+ customers
- Recovery Time Objective (RTO): 4 hours
- Recovery Point Objective (RPO): 6 hours
...


---

# 🏗️ STAGE 3: Build a RAG Agent with LangGraph

Now let's turn this search function into a **tool** that our agent can use.

### Key Insight: RAG as a Tool, Not a Pipeline

There are two ways to use RAG:

| Approach | How it works | When to use |
|----------|-------------|-------------|
| **RAG Pipeline** | ALWAYS search before answering | Simple Q&A over docs |
| **RAG as Tool** | Agent DECIDES when to search | Agent that also does other things |

We'll use **RAG as a tool** — the agent decides whether the question needs
document search or if it can answer directly. This is more flexible and
matches how real agents work.

In [8]:
# ──────────────────────────────────────────────────────
# Build a RAG agent with LangGraph
#
# The agent has ONE tool: search_company_docs
# It decides WHEN to search based on the question.
#
# The system prompt tells it to:
# 1. Use the search tool for company-specific questions
# 2. Cite the source document in its answer
# 3. Say "I don't have information" if search returns nothing
# ──────────────────────────────────────────────────────

from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def search_company_docs(query: str) -> str:
    """Search Acme Corp's internal documents for policies, pricing, SLAs, and HR information.
    Use this tool whenever the user asks about company-specific information.
    Returns relevant document excerpts with source citations."""
    results = search_documents(query, top_k=3)
    if not results:
        return "No relevant documents found."
    
    context = ""
    for r in results:
        context += f"\n\n[Source: {r['source']}]\n{r['content']}"
    return context

# Create the RAG agent
llm = AzureChatOpenAI(
    azure_deployment=MODEL,
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
)

SYSTEM_PROMPT = """You are Acme Corp's internal assistant. You have access to company documents.

RULES:
1. ALWAYS use the search_company_docs tool for questions about Acme Corp policies, pricing, or SLAs
2. ALWAYS cite the source document in your answer (e.g., "According to refund-policy.md...")
3. If the search returns no relevant results, say "I don't have information about that in our documents"
4. NEVER make up information — only use what the documents say
"""

rag_agent = create_react_agent(llm, [search_company_docs], prompt=SYSTEM_PROMPT)

print("✅ RAG agent created")
print("   Tool: search_company_docs (searches Azure AI Search)")
print("   Model: GPT-4.1")
print("   Behavior: searches docs, cites sources, never hallucinates")

✅ RAG agent created
   Tool: search_company_docs (searches Azure AI Search)
   Model: GPT-4.1
   Behavior: searches docs, cites sources, never hallucinates


/var/folders/dt/qn9b7r210m9gxfkwxch5n_sw0000gn/T/ipykernel_5134/278409879.py:48: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  rag_agent = create_react_agent(llm, [search_company_docs], prompt=SYSTEM_PROMPT)


In [9]:
# ──────────────────────────────────────────────────────
# Test the RAG agent with the SAME questions from Stage 1
#
# This time, the agent searches documents first.
# Compare the answers to Stage 1 — they should be
# grounded in actual company data!
# ──────────────────────────────────────────────────────

print("📊 STAGE 3: Agent WITH RAG (access to company docs)")
print("=" * 60)

for q in questions:
    result = rag_agent.invoke({"messages": [("user", q)]})
    answer = result["messages"][-1].content
    
    print(f"\n❓ {q}")
    print(f"🤖 {answer[:300]}")
    print(f"✅ Grounded in actual company documents!")
    print("-" * 60)

📊 STAGE 3: Agent WITH RAG (access to company docs)


k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizedQuery'> and will be ignored



❓ What is Acme Corp's refund policy for enterprise customers?
🤖 According to refund-policy.md, Acme Corp's refund policy for enterprise customers (contracts above $10,000/year) is as follows:

- Full refund within 30 calendar days of contract signing.
- After 30 days: prorated refund minus a 10% early termination fee.
- Annual contracts can be cancelled with 60 
✅ Grounded in actual company documents!
------------------------------------------------------------


k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizedQuery'> and will be ignored



❓ How many vacation days do Acme Corp employees get?
🤖 According to the employee benefits policy, all full-time Acme Corp employees receive 22 vacation days per year. Part-time employees receive vacation days prorated by hours worked. Up to 5 unused vacation days can be carried over to the next year (source: employee-benefits.md).
✅ Grounded in actual company documents!
------------------------------------------------------------


k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizedQuery'> and will be ignored



❓ What is the price of Acme Corp's Professional plan?
🤖 According to pricing-plans.md, the price of Acme Corp's Professional plan is $79 per month. Annual billing is available with a 20% discount.
✅ Grounded in actual company documents!
------------------------------------------------------------


k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizedQuery'> and will be ignored



❓ What is the uptime SLA for Acme Corp Enterprise customers?
🤖 According to the SLA agreement, the uptime SLA for Acme Corp Enterprise customers is 99.9%, which allows for up to 43 minutes of downtime per month (source: sla-agreement.md).
✅ Grounded in actual company documents!
------------------------------------------------------------


### 🤔 Compare Stage 1 vs Stage 3

Go back and compare the answers:

| Question | Stage 1 (No RAG) | Stage 3 (With RAG) |
|----------|------------------|--------------------|
| Refund policy | Guessed generic numbers | Cited exact policy: 14 days standard, 30 days enterprise |
| Vacation days | Made up a number | Correct: 22 days, with 5 carryover |
| Pricing | Invented prices | Correct: $79/month Professional |
| SLA | Generic answer | Correct: 99.9% for Enterprise |

**RAG = the difference between a guessing chatbot and a reliable assistant.**

---

# 🏗️ STAGE 4: Persistent Memory with Cosmos DB

In Lab 01, we used `MemorySaver()` for conversation memory — but that stores
everything in RAM. Restart Python → all conversations lost.

Now let's use **Azure Cosmos DB** for persistent memory.

### Why Cosmos DB?

| Feature | MemorySaver (Lab 01) | Cosmos DB |
|---------|---------------------|----------|
| Storage | RAM | Database (durable) |
| Survives restart | ❌ No | ✅ Yes |
| Multi-server | ❌ No | ✅ Yes |
| Multi-tenant | ❌ No | ✅ Yes (partition by thread_id) |
| Cost | Free | ~$0/month (serverless, pay per request) |
| Production-ready | ❌ | ✅ |

In [16]:
# ──────────────────────────────────────────────────────
# Connect to Cosmos DB and build a simple memory layer
#
# We'll build a lightweight wrapper that:
# 1. Uses MemorySaver for the LangGraph checkpointing (conversation state)
# 2. Writes conversation SUMMARIES to Cosmos DB (persistent user memory)
#
# This gives us BOTH:
# - Fast in-process memory for the ReAct loop (MemorySaver)
# - Durable storage that survives restarts (Cosmos DB)
#
# In production, you'd use a full CosmosDBSaver that
# implements BaseCheckpointSaver — but this pattern
# shows the concept clearly.
# ──────────────────────────────────────────────────────

from azure.cosmos import CosmosClient
from langgraph.checkpoint.memory import MemorySaver
from datetime import datetime

cosmos_endpoint = os.getenv("AZURE_COSMOS_ENDPOINT")
cosmos_key = os.getenv("AZURE_COSMOS_KEY")
cosmos_db_name = os.getenv("AZURE_COSMOS_DATABASE", "agent-platform")

cosmos_available = False

if cosmos_endpoint and cosmos_key:
    cosmos_client = CosmosClient(url=cosmos_endpoint, credential=cosmos_key)
    db = cosmos_client.get_database_client(cosmos_db_name)
    threads_container = db.get_container_client("threads")
    cosmos_available = True
    
    print(f"✅ Connected to Cosmos DB")
    print(f"   Database: {cosmos_db_name}")
    print(f"   Container: threads")
else:
    print("⚠️ Cosmos DB not configured. Using MemorySaver only.")

def save_conversation_to_cosmos(thread_id: str, messages: list, summary: str = ""):
    """Save a conversation summary to Cosmos DB for persistent storage."""
    if not cosmos_available:
        return
    
    doc = {
        "id": f"{thread_id}-{datetime.now().strftime('%Y%m%d%H%M%S')}",
        "thread_id": thread_id,
        "timestamp": datetime.now().isoformat(),
        "message_count": len(messages),
        "summary": summary,
        "last_user_message": next(
            (m.content for m in reversed(messages) if hasattr(m, 'type') and m.type == 'human'),
            ""
        ),
        "last_assistant_message": next(
            (m.content[:500] for m in reversed(messages) if hasattr(m, 'type') and m.type == 'ai'),
            ""
        ),
    }
    
    threads_container.upsert_item(doc)
    return doc["id"]

# Test Cosmos DB write
if cosmos_available:
    test_id = save_conversation_to_cosmos(
        "test-thread",
        [],
        summary="Test connection"
    )
    print(f"\n✅ Cosmos DB write test: saved document '{test_id}'")
    print(f"   Conversations will be persisted to Cosmos DB!")
else:
    print("\n💡 Without Cosmos DB, memory only lasts while this notebook is running.")

✅ Connected to Cosmos DB
   Database: agent-platform
   Container: threads

✅ Cosmos DB write test: saved document 'test-thread-20260316095405'
   Conversations will be persisted to Cosmos DB!


---

# 🏗️ STAGE 5: The Complete Agent — RAG + Memory + Cosmos DB

Now let's combine everything into a production-quality agent:
- **RAG tool** → searches company documents via Azure AI Search
- **MemorySaver** → conversation state within a session (fast, in-process)
- **Cosmos DB** → persistent storage of conversation summaries (survives restarts)
- **System prompt** → grounding rules, source citation

### Two layers of memory

In production agents, memory works in **two layers**:

```
┌─────────────────────────────────────────────────────┐
│  LAYER 1: Fast Memory (MemorySaver / Redis)         │
│  • Full conversation state for the ReAct loop       │
│  • In-process, fast read/write                      │
│  • Lost on restart (that's OK — it's a cache)       │
│                                                      │
│  LAYER 2: Durable Memory (Cosmos DB / Postgres)     │
│  • Conversation summaries, user facts               │
│  • Persists forever, survives restarts               │
│  • Loaded on startup for new sessions               │
└─────────────────────────────────────────────────────┘
```

In [17]:
# ──────────────────────────────────────────────────────
# THE COMPLETE AGENT: RAG + Memory + Cosmos DB
#
# This agent:
# 1. Searches company documents when needed (RAG via Azure AI Search)
# 2. Remembers conversation within session (MemorySaver)
# 3. Persists conversation summaries to Cosmos DB (durable)
# 4. Cites sources in its answers
# 5. Never makes up information
# ──────────────────────────────────────────────────────

memory = MemorySaver()

complete_agent = create_react_agent(
    llm,
    [search_company_docs],
    prompt=SYSTEM_PROMPT,
    checkpointer=memory
)

# Thread ID for this conversation
THREAD_ID = "employee-roi-456"
config = {"configurable": {"thread_id": THREAD_ID}}

# ── Conversation 1: Ask about benefits ──
print("💬 Conversation 1: Ask about benefits")
r1 = complete_agent.invoke(
    {"messages": [("user", "I'm a new employee. How many vacation days do I get?")]},
    config=config
)
print(f"🤖 {r1['messages'][-1].content[:300]}")

# Save to Cosmos DB
if cosmos_available:
    save_conversation_to_cosmos(THREAD_ID, r1["messages"], "New employee asking about vacation days")
    print("   💾 Saved to Cosmos DB")

print(f"\n{'─' * 60}")

# ── Conversation 2: Follow-up (agent remembers!) ──
print("💬 Conversation 2: Follow-up (agent remembers context!)")
r2 = complete_agent.invoke(
    {"messages": [("user", "Can I carry over unused days? And what about remote work?")]},
    config=config
)
print(f"🤖 {r2['messages'][-1].content[:300]}")

if cosmos_available:
    save_conversation_to_cosmos(THREAD_ID, r2["messages"], "Follow-up on vacation carryover and remote work")
    print("   💾 Saved to Cosmos DB")

print(f"\n{'─' * 60}")

# ── Conversation 3: Different topic (still remembers!) ──
print("💬 Conversation 3: Different topic (agent still remembers!)")
r3 = complete_agent.invoke(
    {"messages": [("user", "My manager asked about our SLA. What's the uptime guarantee for Enterprise?")]},
    config=config
)
print(f"🤖 {r3['messages'][-1].content[:300]}")

if cosmos_available:
    save_conversation_to_cosmos(THREAD_ID, r3["messages"], "Manager needs Enterprise SLA info")
    print("   💾 Saved to Cosmos DB")

print(f"\n{'─' * 60}")
print(f"\n🎉 The agent searched docs, remembered the conversation, AND saved to Cosmos DB!")

/var/folders/dt/qn9b7r210m9gxfkwxch5n_sw0000gn/T/ipykernel_5134/1056672506.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  complete_agent = create_react_agent(


💬 Conversation 1: Ask about benefits


k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizedQuery'> and will be ignored


🤖 According to the employee benefits policy, all full-time employees at Acme Corp receive 22 vacation days per year. If you are part-time, your vacation days are prorated based on the hours you work. Up to 5 unused vacation days can be carried over to the next year. Vacation requests must be submitted
   💾 Saved to Cosmos DB

────────────────────────────────────────────────────────────
💬 Conversation 2: Follow-up (agent remembers context!)
🤖 Yes, you can carry over unused vacation days—up to 5 days can be carried over to the next year.

Regarding remote work, employees may work remotely up to 3 days per week. You must be available on Microsoft Teams during core hours: 10:00 AM – [end time not specified in excerpt] (employee-benefits.md)
   💾 Saved to Cosmos DB

────────────────────────────────────────────────────────────
💬 Conversation 3: Different topic (agent still remembers!)


k_nearest_neighbors is not a known attribute of class <class 'azure.search.documents._generated.models._models_py3.VectorizedQuery'> and will be ignored


🤖 According to the Service Level Agreement, the uptime guarantee for Enterprise customers is 99.9%, which allows for up to 43 minutes of downtime per month (sla-agreement.md).
   💾 Saved to Cosmos DB

────────────────────────────────────────────────────────────

🎉 The agent searched docs, remembered the conversation, AND saved to Cosmos DB!


### 🔍 Verify: What's Actually in Cosmos DB?

Let's query Cosmos DB to see what was saved.
This data persists **forever** — even if you restart Python, close VS Code, or reboot your machine.

In [18]:
# ──────────────────────────────────────────────────────
# VERIFY: What's actually stored in Cosmos DB?
#
# This proves the data is persisted — you can restart
# the notebook and this data will still be there.
#
# Note: You must run Stage 4 and Stage 5 cells first!
# ──────────────────────────────────────────────────────

try:
    _cosmos_ok = cosmos_available
except NameError:
    # If cosmos_available isn't defined, Stage 4 wasn't run
    print("⚠️ Please run Stage 4 (Cosmos DB connection) first!")
    print("   Run the cells in order: Stage 4 → Stage 5 → this cell.")
    _cosmos_ok = False

if _cosmos_ok:
    print("🔍 Querying Cosmos DB for saved conversations...")
    print("=" * 60)
    
    query = f"SELECT * FROM c WHERE c.thread_id = '{THREAD_ID}' ORDER BY c.timestamp DESC"
    items = list(threads_container.query_items(query=query, enable_cross_partition_query=True))
    
    print(f"\n📋 Found {len(items)} records for thread '{THREAD_ID}':\n")
    for item in items:
        print(f"  📝 {item.get('timestamp', 'unknown')}")
        print(f"     Summary: {item.get('summary', 'N/A')}")
        print(f"     Messages: {item.get('message_count', 0)}")
        print(f"     Last question: {item.get('last_user_message', '')[:80]}")
        print()
    
    print("💡 This data survives restarts. In production, load it on startup")
    print("   to restore conversation context for returning users.")
elif not _cosmos_ok and 'cosmos_available' in dir():
    print("⚠️ Cosmos DB not available — conversations only in RAM.")

🔍 Querying Cosmos DB for saved conversations...

📋 Found 3 records for thread 'employee-roi-456':

  📝 2026-03-16T09:54:17.636518
     Summary: Manager needs Enterprise SLA info
     Messages: 10
     Last question: My manager asked about our SLA. What's the uptime guarantee for Enterprise?

  📝 2026-03-16T09:54:15.684499
     Summary: Follow-up on vacation carryover and remote work
     Messages: 6
     Last question: Can I carry over unused days? And what about remote work?

  📝 2026-03-16T09:54:13.502034
     Summary: New employee asking about vacation days
     Messages: 4
     Last question: I'm a new employee. How many vacation days do I get?

💡 This data survives restarts. In production, load it on startup
   to restore conversation context for returning users.


---

# 🎓 Summary: What We Built and Learned

### The Five Stages

| Stage | What We Did | Key Insight |
|-------|------------|-------------|
| **Stage 1** | Asked without RAG → hallucination | LLMs guess when they don't have data |
| **Stage 2** | Built RAG pipeline | Chunk → Embed → Index → Search |
| **Stage 3** | RAG agent with tool | Agent decides WHEN to search |
| **Stage 4** | Cosmos DB memory | Production-grade persistence |
| **Stage 5** | Complete agent | RAG + Memory = production quality |

### Key Concepts

| Concept | What It Means |
|---------|---------------|
| **RAG** | Search your docs, add to prompt → grounded answers |
| **Embedding** | Text → numbers (vector) for similarity search |
| **Chunking** | Split docs into searchable pieces |
| **Hybrid search** | Vector + keyword search combined |
| **Grounding** | Answer based on data, not imagination |
| **Persistent memory** | Conversations stored in database |
| **RAG as a tool** | Agent decides when to search |

### What's Next?

In **Lab 04**, we'll explore **orchestration patterns** — sequential, parallel,
and map-reduce workflows with multiple agents.

---

> **[← Back to Lab 02](../lab-02-model-routing/README.md)** | **[→ Lab 04: Orchestration Patterns](../lab-04-orchestration/README.md)**